# 05 Model architecture
Masked PointNet / PointNet++ encoder -> aggregation -> template-conditioned decoder -> Gaussian head (bounded log-variance).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)

In [ ]:
import yaml, torch
from src.data import dataset as ds, preprocessing as pp
from src.models.uncertaface import UncertaFace
R = 'UNCERTAFACE_DATA'; norm = pp.Normalizer.load(R + '/metadata/normalizer.json')
tpl = ds.fit_template(R, norm, pp.read_split(R, 'train')['base_ids'])
cfg = yaml.safe_load(open('configs/model.yaml'))['model']
m = UncertaFace(cfg, tpl, norm.scale)
print('parameters:', m.num_parameters())
b = next(iter(ds.make_loader(ds.UncertafaceDataset(R, 'validation', norm), 4, False)))
out = m.forward_batch(b)
print({k: tuple(v.shape) for k, v in out.items()})

In [ ]:
from src.uncertainty.estimation import gaussian_to_mm
phys = gaussian_to_mm(out['mean'], out['logvar'], norm)
print('initial predicted per-axis std (mm):', float(phys['std_mm'].mean()))